IMPORTS

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path(os.path.abspath('')).resolve().parents[2]
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import nvitk.core as core
core.setup(globals())

import nvitk as nv

In [ ]:
tof = nv.imread('/home/imarcoss/DATA/LabVF/PESA-Brain/WVI-BB/RESULTS/eicab_test/PESA15689521/TOF_resampled.nii.gz')
eicab = nv.imread('/home/imarcoss/DATA/LabVF/PESA-Brain/WVI-BB/RESULTS/eicab_test/PESA15689521/TOF_eICAB_CW.nii.gz')

OSError: [Errno 5] Input/output error

CODE

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu

from nvitk.core.array import to_numpy
from nvitk.morphology.centerline import compute_centerlines
from nvitk.morphology.components import remove_small_components_by_fraction
from nvitk.pipes.pesa_brain.black_blood.labels import (
    BB_LICA,
    BB_RICA,
    BB_ARTERIAL_LABEL_IDS,
    bb_vessel_name,
    relabel_eicab_to_bb,
)
from nvitk.pipes.pesa_brain.black_blood.util.centerlines_from_eicab import rasterize_centerlines_mask

ICA_IDS = (BB_LICA, BB_RICA)
CROP_PAD = 5
CL_BARRIER_RADIUS = 2
MIN_COMPONENT_FRAC = 0.005


def bbox_with_padding(roi: np.ndarray, shape, pad: int):
    xs, ys, zs = np.nonzero(roi)
    if xs.size == 0:
        return None
    nx, ny, nz = shape
    p = max(0, int(pad))
    return (
        max(0, int(xs.min()) - p),
        min(nx - 1, int(xs.max()) + p),
        max(0, int(ys.min()) - p),
        min(ny - 1, int(ys.max()) + p),
        max(0, int(zs.min()) - p),
        min(nz - 1, int(zs.max()) + p),
    )


def local_otsu_mask(wvi_crop: np.ndarray, *, intensity_max: float) -> tuple[np.ndarray, float]:
    """TOF lumen is hypointense: invert, then Otsu inside the crop."""
    inv = intensity_max - wvi_crop.astype(np.float64)
    pos = inv[inv > 0]
    if pos.size < 2:
        return np.zeros(wvi_crop.shape, dtype=bool), np.nan
    t = float(threshold_otsu(pos))
    mask = inv > t
    if MIN_COMPONENT_FRAC > 0:
        mask = remove_small_components_by_fraction(
            mask, min_fraction=MIN_COMPONENT_FRAC, connectivity=1
        )
    return mask, t


def dilated_other_centerlines(clm: np.ndarray, label_id: int, bbox, radius: int) -> np.ndarray:
    from scipy import ndimage

    i0, i1, j0, j1, k0, k1 = bbox
    other = (clm != 0) & (clm != int(label_id))
    if radius > 0 and np.any(other):
        other = ndimage.binary_dilation(other, iterations=radius)
    return other[i0 : i1 + 1, j0 : j1 + 1, k0 : k1 + 1]


# --- inputs (same grid as TOF) ---
wvi = to_numpy(tof.data).astype(np.float32)
eicab_np = to_numpy(eicab.data).astype(np.int32)
shape = tuple(int(s) for s in wvi.shape[:3])
intensity_max = float(wvi.max()) or 1.0

# eICAB → BB labels, skeleton centerlines, rasterized mask
bb = relabel_eicab_to_bb(eicab_np)
centerlines = compute_centerlines(
    bb, labels=sorted(BB_ARTERIAL_LABEL_IDS), min_points=5
)
cl_mask = rasterize_centerlines_mask(shape, centerlines)

# --- ICA-only crop-resegment (flow-style: bbox + local Otsu + CL barriers) ---
seg_ica = np.zeros(shape, dtype=np.int32)
stats = []

for lid in ICA_IDS:
    roi = cl_mask == lid
    bbox = bbox_with_padding(roi, shape, CROP_PAD)
    if bbox is None:
        stats.append({"label": bb_vessel_name(lid), "n_voxels": 0, "warning": "no centerline"})
        continue

    i0, i1, j0, j1, k0, k1 = bbox
    crop = wvi[i0 : i1 + 1, j0 : j1 + 1, k0 : k1 + 1]
    mask, t = local_otsu_mask(crop, intensity_max=intensity_max)

    forbidden = dilated_other_centerlines(cl_mask, lid, bbox, CL_BARRIER_RADIUS)
    slab = seg_ica[i0 : i1 + 1, j0 : j1 + 1, k0 : k1 + 1]
    write = mask & (slab == 0) & ~forbidden
    n = int(np.count_nonzero(write))
    slab[write] = int(lid)

    stats.append(
        {
            "label": bb_vessel_name(lid),
            "bbox": bbox,
            "otsu_thresh": t,
            "n_voxels": n,
            "n_centerline_pts": int(centerlines.get(lid, np.empty((0, 3))).shape[0]),
        }
    )

print("Centerlines:")
for lid in sorted(centerlines):
    print(f"  {bb_vessel_name(lid):8s}  {centerlines[lid].shape[0]:4d} pts")

print("\nICA resegmentation:")
for st in stats:
    print(st)

# quick axial check (mid Z)
k = shape[2] // 2
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(wvi[:, :, k].T, cmap="gray", origin="lower")
axes[0].set_title("TOF")
axes[1].imshow(bb[:, :, k].T, origin="lower")
axes[1].set_title("eICAB → BB")
overlay = np.ma.masked_where(seg_ica[:, :, k] == 0, seg_ica[:, :, k])
axes[2].imshow(wvi[:, :, k].T, cmap="gray", origin="lower")
axes[2].imshow(overlay.T, origin="lower", alpha=0.55, cmap="cool", vmin=1, vmax=2)
axes[2].set_title("ICA reseg (bbox + Otsu)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()